# LeetCode #187 — Repeated DNA Sequences: cửa sổ trượt + bảng băm, mã hóa 2 bit

**Mức độ:** Medium · **Chủ đề:** Hash Table, String, Bit Manipulation, Sliding Window, Rolling Hash

Skeleton trên LeetCode:

```python
class Solution:
    def findRepeatedDnaSequences(self, s: str) -> List[str]:
```

Mục tiêu của notebook:

1. Duyệt mọi **chuỗi con độ dài 10** bằng cửa sổ trượt; dùng **set** để biết đã gặp chưa.
2. Tránh trả trùng: hai set `seen` và `repeated`.
3. Tối ưu: mã hóa mỗi nucleotide bằng **2 bit** → mỗi cửa sổ là một số nguyên 20 bit, cập nhật `O(1)` khi trượt (rolling hash "hoàn hảo", không va chạm).
4. Tổng quát: **rolling hash** (Rabin–Karp) cho bảng chữ cái bất kỳ.

# 1. Bài toán là gì?

Chuỗi DNA gồm các ký tự `'A'`, `'C'`, `'G'`, `'T'`. Cho chuỗi `s`, trả về mọi chuỗi con **dài 10** xuất hiện **nhiều hơn một lần** trong `s`. Thứ tự tùy ý.

**Constraints (theo đề):**

- `1 <= s.length <= 10^5`
- `s[i]` là `'A'`, `'C'`, `'G'` hoặc `'T'`.

---

## Ví dụ

```text
s = "AAAAACCCCCAAAAACCCCCCAAAAAGGGTTT"   ->  ["AAAAACCCCC", "CCCCCAAAAA"]
s = "AAAAAAAAAAAAA"                      ->  ["AAAAAAAAAA"]
```

(Các lần xuất hiện **được phép chồng lên nhau**, như ví dụ 2.)

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Bảng băm (hash table) — `dict` và `set`
2. Cửa sổ trượt (sliding window)
3. Biểu diễn nhị phân và các phép toán bit

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [1]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [2]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 536.89 ms | trên set: 0.077 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [3]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [4]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.2 Cửa sổ trượt (sliding window)

### Ý tưởng

Cửa sổ trượt là trường hợp riêng của hai con trỏ cùng chiều: hai đầu `left`, `right` xác định một **đoạn liên tiếp** `[left, right]`. Ta **mở rộng** bên phải từng bước; khi cửa sổ vi phạm điều kiện, **thu hẹp** bên trái cho tới khi hợp lệ lại. Mỗi phần tử vào cửa sổ một lần, ra một lần → `O(n)`.

```text
Đoạn con dài nhất có tổng <= 7 trong [2, 1, 5, 1, 3, 2]
[2]           tổng 2   ✓  dài 1
[2,1]         tổng 3   ✓  dài 2
[2,1,5]       tổng 8   ✗  → bỏ 2 → [1,5] tổng 6 ✓
[1,5,1]       tổng 7   ✓  dài 3
[1,5,1,3]     tổng 10  ✗  → bỏ 1, bỏ 5 → [1,3] tổng 4
[1,3,2]       tổng 6   ✓  dài 3
```

### Khung (cửa sổ thay đổi kích thước)

```text
left = 0
for right in range(n):
    thêm a[right] vào cửa sổ
    while cửa sổ không hợp lệ:
        bỏ a[left] khỏi cửa sổ; left += 1
    cập nhật đáp án với cửa sổ [left, right]
```

In [5]:
def dai_nhat_tong_khong_qua(a, k):
    left, tong, best = 0, 0, 0
    for right, x in enumerate(a):
        tong += x
        while tong > k:
            tong -= a[left]
            left += 1
        best = max(best, right - left + 1)
        print(f"   right={right} cửa sổ={a[left:right + 1]} tổng={tong}")
    return best

print("→ dài nhất:", dai_nhat_tong_khong_qua([2, 1, 5, 1, 3, 2], 7))

   right=0 cửa sổ=[2] tổng=2
   right=1 cửa sổ=[2, 1] tổng=3
   right=2 cửa sổ=[1, 5] tổng=6
   right=3 cửa sổ=[1, 5, 1] tổng=7
   right=4 cửa sổ=[1, 3] tổng=4
   right=5 cửa sổ=[1, 3, 2] tổng=6
→ dài nhất: 3


### Cửa sổ với bảng băm: đoạn không có ký tự lặp

Cửa sổ giữ **tập ký tự** bên trong. Khi ký tự mới đã có trong cửa sổ, thu hẹp trái cho tới khi bản cũ bị bỏ ra.

In [6]:
def dai_nhat_khong_lap(s):
    trong, left, best = set(), 0, 0
    for right, ch in enumerate(s):
        while ch in trong:
            trong.remove(s[left])
            left += 1
        trong.add(ch)
        best = max(best, right - left + 1)
    return best

for s_sw in ["abcabcbb", "bbbbb", "pwwkew", ""]:
    print(f"{s_sw!r:>11}: {dai_nhat_khong_lap(s_sw)}")

 'abcabcbb': 3
    'bbbbb': 1
   'pwwkew': 3
         '': 0


### Lỗi hay gặp

- Dùng `if` thay cho `while` khi thu hẹp → cửa sổ có thể vẫn chưa hợp lệ.
- Cập nhật đáp án **trước** khi làm cửa sổ hợp lệ.
- Áp dụng cho bài mà thu hẹp không giữ tính đơn điệu (ví dụ tổng với số **âm**: bỏ bớt phần tử chưa chắc làm tổng giảm).

## 2.3 Biểu diễn nhị phân và các phép toán bit

### Số nguyên dưới dạng bit

Máy tính lưu số nguyên ở hệ **nhị phân**: mỗi chữ số (bit) là 0 hoặc 1, bit thứ `k` (đếm từ 0, bên phải) có giá trị `2^k`.

```text
13 = 8 + 4 + 1 = 1·2³ + 1·2² + 0·2¹ + 1·2⁰ = 0b1101
bit:              3     2      1      0
```

In [7]:
x_bit = 13
print("bin(13) =", bin(x_bit), "| format 8 bit:", format(x_bit, "08b"), "| int('1101', 2) =", int("1101", 2))
print("bit thứ k của 13:", [(x_bit >> k) & 1 for k in range(4)], "(từ bit 0)")

bin(13) = 0b1101 | format 8 bit: 00001101 | int('1101', 2) = 13
bit thứ k của 13: [1, 0, 1, 1] (từ bit 0)


### Sáu phép toán bit

| Phép | Ký hiệu | Quy tắc từng bit | Ví dụ `12 = 1100`, `10 = 1010` |
|---|---|---|---|
| AND | `a & b` | 1 nếu **cả hai** là 1 | `1000` = 8 |
| OR | `a \| b` | 1 nếu **ít nhất một** là 1 | `1110` = 14 |
| XOR | `a ^ b` | 1 nếu **khác nhau** | `0110` = 6 |
| NOT | `~a` | đảo mọi bit | `~12 = -13` (Python) |
| dịch trái | `a << k` | thêm `k` số 0 bên phải = nhân `2^k` | `12 << 1 = 24` |
| dịch phải | `a >> k` | bỏ `k` bit phải = chia nguyên `2^k` | `12 >> 2 = 3` |

In [8]:
a_b, b_b = 12, 10
for ten, v in [("a & b", a_b & b_b), ("a | b", a_b | b_b), ("a ^ b", a_b ^ b_b), ("a << 1", a_b << 1), ("a >> 2", a_b >> 2), ("~a", ~a_b)]:
    print(f"{ten:<7} = {v:>4}   bin: {bin(v)}")

a & b   =    8   bin: 0b1000
a | b   =   14   bin: 0b1110
a ^ b   =    6   bin: 0b110
a << 1  =   24   bin: 0b11000
a >> 2  =    3   bin: 0b11
~a      =  -13   bin: -0b1101


### Các mẹo hay dùng

| Mục đích | Biểu thức |
|---|---|
| Kiểm tra bit `k` | `(x >> k) & 1` |
| Bật / tắt / lật bit `k` | `x \| (1 << k)`, `x & ~(1 << k)`, `x ^ (1 << k)` |
| Số chẵn? | `x & 1 == 0` |
| Bỏ bit 1 thấp nhất | `x & (x - 1)` |
| Lấy bit 1 thấp nhất | `x & -x` |
| Là lũy thừa của 2? | `x > 0 and x & (x - 1) == 0` |

**Tính chất XOR** (cực kỳ hay dùng): `x ^ x = 0`, `x ^ 0 = x`, giao hoán và kết hợp → XOR cả dãy thì các số xuất hiện **chẵn** lần triệt tiêu nhau.

In [9]:
x_m = 0b10110
print("bỏ bit 1 thấp nhất:", format(x_m, "05b"), "→", format(x_m & (x_m - 1), "05b"))
print("lấy bit 1 thấp nhất:", format(x_m & -x_m, "05b"))
print("lũy thừa của 2:", [v for v in range(1, 40) if v & (v - 1) == 0])
print("đếm bit 1 của 255:", bin(255).count("1"))
xor_all = 0
for v in [4, 1, 2, 1, 2]:
    xor_all ^= v
print("XOR [4,1,2,1,2] =", xor_all, "(số xuất hiện lẻ lần)")

bỏ bit 1 thấp nhất: 10110 → 10100
lấy bit 1 thấp nhất: 00010
lũy thừa của 2: [1, 2, 4, 8, 16, 32]
đếm bit 1 của 255: 8
XOR [4,1,2,1,2] = 4 (số xuất hiện lẻ lần)


### Dùng bit làm tập hợp (bitmask)

Với `n` phần tử, một số `mask` từ `0` tới `2^n − 1` biểu diễn một **tập con**: bit `i` bằng 1 nghĩa là lấy phần tử `i`. Duyệt mọi `mask` = duyệt mọi tập con.

In [10]:
items = ["a", "b", "c"]
for mask in range(1 << len(items)):
    tap = [items[i] for i in range(len(items)) if mask >> i & 1]
    print(format(mask, "03b"), tap)

000 []
001 ['a']
010 ['b']
011 ['a', 'b']
100 ['c']
101 ['a', 'c']
110 ['b', 'c']
111 ['a', 'b', 'c']


### Số âm và giới hạn 32 bit

Ngôn ngữ như C/Java lưu `int` bằng 32 bit, số âm dạng **bù hai** (two's complement): `-x = ~x + 1`. Python có số nguyên **không giới hạn**, nên muốn mô phỏng 32 bit phải **che** bằng `& 0xFFFFFFFF` rồi đổi lại thành số có dấu.

In [11]:
def thanh_32bit(x):
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >> 31 else x       # bit 31 là bit dấu

print("-1 ở 32 bit:", format(-1 & 0xFFFFFFFF, "032b"))
print("2^31 tràn thành:", thanh_32bit(2 ** 31), "| (2^31 - 1) + 1 =", thanh_32bit(2 ** 31 - 1 + 1))

-1 ở 32 bit: 11111111111111111111111111111111
2^31 tràn thành: -2147483648 | (2^31 - 1) + 1 = -2147483648


Lỗi hay gặp: nhầm `^` là lũy thừa (lũy thừa là `**`); quên ngoặc vì `&`, `|`, `^` có độ ưu tiên **thấp hơn** `==` — viết `(x & 1) == 0`; `~x` trong Python là `-x - 1`, không phải "đảo 32 bit".

## 2.4 Áp dụng vào bài này

Xét mọi đoạn con dài 10 — một **cửa sổ trượt** kích thước cố định — và đếm bằng **bảng băm** (`set` đã gặp + `set` kết quả). Tối ưu: mỗi nucleotide mã hóa bằng **2 bit**, một đoạn dài 10 thành một số 20 bit; trượt cửa sổ bằng `((h << 2) | code) & mask` thay vì cắt chuỗi mới mỗi bước.

# 3. Nền tảng

In [12]:
from typing import List
from collections import Counter
import random, time

s = "AAAAACCCCCAAAAACCCCCCAAAAAGGGTTT"
L = 10
print("số cửa sổ độ dài 10:", len(s) - L + 1)
print("ba cửa sổ đầu:", [s[i:i + L] for i in range(3)])

số cửa sổ độ dài 10: 23
ba cửa sổ đầu: ['AAAAACCCCC', 'AAAACCCCCA', 'AAACCCCCAA']


Có `n − 9` cửa sổ (nếu `n >= 10`). Mỗi lát cắt `s[i:i+10]` tạo chuỗi mới dài 10 → `O(10)`.

# 4. Baseline: so mọi cặp cửa sổ

`O(n²)` cặp, mỗi cặp so 10 ký tự — với `n = 10^5` là quá chậm.

In [13]:
def repeated_brute(s: str) -> List[str]:
    out = set()
    subs = [s[i:i + 10] for i in range(len(s) - 9)]
    for i in range(len(subs)):
        for j in range(i + 1, len(subs)):
            if subs[i] == subs[j]:
                out.add(subs[i])
    return sorted(out)

print(repeated_brute("AAAAACCCCCAAAAACCCCCCAAAAAGGGTTT"))

['AAAAACCCCC', 'CCCCCAAAAA']


# 5. Ý tưởng chính: cửa sổ trượt + hai set

Duyệt từng cửa sổ một lần:

- Chưa có trong `seen` → thêm vào `seen`.
- Đã có trong `seen` → thêm vào `repeated` (là `set` nên dù gặp lần ba, lần tư vẫn chỉ một bản).

In [14]:
def repeated_sets(s: str) -> List[str]:
    seen, repeated = set(), set()
    for i in range(len(s) - 9):
        sub = s[i:i + 10]
        if sub in seen:
            repeated.add(sub)
        else:
            seen.add(sub)
    return list(repeated)

print(sorted(repeated_sets("AAAAACCCCCAAAAACCCCCCAAAAAGGGTTT")), repeated_sets("AAAAAAAAAAAAA"))

['AAAAACCCCC', 'CCCCCAAAAA'] ['AAAAAAAAAA']


`O(10 · n)` thời gian và bộ nhớ (mỗi chuỗi con 10 ký tự được lưu) — đã đủ nhanh. Nhưng có cách gọn hơn về bộ nhớ.

# 6. Tối ưu: mã hóa 2 bit — cửa sổ thành một số 20 bit

Chỉ có 4 ký tự → mỗi ký tự cần **2 bit**: `A = 00`, `C = 01`, `G = 10`, `T = 11`. Cửa sổ 10 ký tự = **20 bit** = một số nguyên `< 2^20`. Hai cửa sổ bằng nhau ⟺ hai số bằng nhau (mã hóa một-một, **không có va chạm**).

**Trượt cửa sổ trong `O(1)`:**

```text
code = ((code << 2) | bit(ký tự mới)) & MASK        MASK = 2^20 − 1 = 0xFFFFF
```

- `code << 2`: dời các ký tự cũ sang trái 2 bit, chừa chỗ cho ký tự mới.
- `| bit`: ghép ký tự mới vào 2 bit thấp nhất.
- `& MASK`: chỉ giữ 20 bit thấp → ký tự cũ nhất (bị đẩy ra khỏi 20 bit) biến mất.

In [15]:
BITS = {"A": 0, "C": 1, "G": 2, "T": 3}
MASK = (1 << 20) - 1

def encode(sub: str) -> int:
    code = 0
    for ch in sub:
        code = (code << 2) | BITS[ch]
    return code

print("encode('ACGT') =", encode("ACGT"), "=", format(encode("ACGT"), "08b"))

def repeated_bits(s: str) -> List[str]:
    if len(s) < 10:
        return []
    seen, added, out = set(), set(), []
    code = 0
    for i, ch in enumerate(s):
        code = ((code << 2) | BITS[ch]) & MASK
        if i >= 9:                               # đã đủ 10 ký tự trong cửa sổ
            if code in seen and code not in added:
                out.append(s[i - 9:i + 1])
                added.add(code)
            seen.add(code)
    return out

print(sorted(repeated_bits("AAAAACCCCCAAAAACCCCCCAAAAAGGGTTT")), repeated_bits("AAAAAAAAAAAAA"))

encode('ACGT') =

 27 = 00011011
['AAAAACCCCC', 'CCCCCAAAAA'] ['AAAAAAAAAA']


# 7. Mô phỏng: mã số của các cửa sổ

In [16]:
def trace_bits(s, show=8):
    code = 0
    seen = {}
    for i, ch in enumerate(s):
        code = ((code << 2) | BITS[ch]) & MASK
        if i >= 9:
            w = s[i - 9:i + 1]
            note = f"gặp lại (lần đầu ở vị trí {seen[code]})" if code in seen else ""
            seen.setdefault(code, i - 9)
            if i - 9 < show or note:
                print(f"   cửa sổ {i - 9:>2}: {w}  code = {code:>7} = {format(code, '020b')} {note}")

trace_bits("AAAAACCCCCAAAAACCCCCCAAAAAGGGTTT")

   cửa sổ  0: AAAAACCCCC  code =     341 = 00000000000101010101 


   cửa sổ  1: AAAACCCCCA  code =    1364 = 00000000010101010100 
   cửa sổ  2: AAACCCCCAA  code =    5456 = 00000001010101010000 
   cửa sổ  3: AACCCCCAAA  code =   21824 = 00000101010101000000 
   cửa sổ  4: ACCCCCAAAA  code =   87296 = 00010101010100000000 
   cửa sổ  5: CCCCCAAAAA  code =  349184 = 01010101010000000000 
   cửa sổ  6: CCCCAAAAAC  code =  348161 = 01010101000000000001 
   cửa sổ  7: CCCAAAAACC  code =  344069 = 01010100000000000101 
   cửa sổ 10: AAAAACCCCC  code =     341 = 00000000000101010101 gặp lại (lần đầu ở vị trí 0)
   cửa sổ 16: CCCCCAAAAA  code =  349184 = 01010101010000000000 gặp lại (lần đầu ở vị trí 5)


# 8. Phiên bản đúng format LeetCode

In [17]:
class Solution:
    def findRepeatedDnaSequences(self, s: str) -> List[str]:
        if len(s) < 10:
            return []
        bits = {"A": 0, "C": 1, "G": 2, "T": 3}
        mask = (1 << 20) - 1
        seen, added, out = set(), set(), []
        code = 0
        for i, ch in enumerate(s):
            code = ((code << 2) | bits[ch]) & mask
            if i >= 9:
                if code in seen:
                    if code not in added:
                        added.add(code)
                        out.append(s[i - 9:i + 1])
                else:
                    seen.add(code)
        return out

In [18]:
def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    print(f"{name:<12} PASS  {sorted(actual)}")


sol = Solution()
check("Ví dụ 1", sol.findRepeatedDnaSequences("AAAAACCCCCAAAAACCCCCCAAAAAGGGTTT"), ["AAAAACCCCC", "CCCCCAAAAA"])
check("Ví dụ 2", sol.findRepeatedDnaSequences("AAAAAAAAAAAAA"), ["AAAAAAAAAA"])
check("Ngắn", sol.findRepeatedDnaSequences("ACGT"), [])
check("Đúng 10", sol.findRepeatedDnaSequences("ACGTACGTAC"), [])
check("Ba lần", sol.findRepeatedDnaSequences("ACGTACGTAC" * 3), ["ACGTACGTAC", "CGTACGTACA", "GTACGTACAC", "TACGTACACG", "ACGTACACGT", "CGTACACGTA", "GTACACGTAC", "TACACGTACG", "ACACGTACGT", "CACGTACGTA"])

random.seed(187)
for _ in range(500):
    t = "".join(random.choice("AC") for _ in range(random.randint(1, 40)))
    exp = repeated_brute(t)
    assert sorted(sol.findRepeatedDnaSequences(t)) == exp == sorted(repeated_sets(t)), t
print("500 chuỗi ngẫu nhiên: khớp brute force")

big = "".join(random.choice("ACGT") for _ in range(10**5))
for name, f in [("mã 2 bit", sol.findRepeatedDnaSequences), ("set chuỗi", repeated_sets)]:
    t0 = time.time(); r = f(big); t1 = time.time()
    print(f"n = 10^5, {name:<9}: {t1 - t0:.3f}s, {len(r)} chuỗi lặp")

Ví dụ 1      PASS  ['AAAAACCCCC', 'CCCCCAAAAA']


Ví dụ 2      PASS  ['AAAAAAAAAA']
Ngắn         PASS  []
Đúng 10      PASS  []
Ba lần       PASS  ['ACACGTACGT', 'ACGTACACGT', 'ACGTACGTAC', 'CACGTACGTA', 'CGTACACGTA', 'CGTACGTACA', 'GTACACGTAC', 'GTACGTACAC', 'TACACGTACG', 'TACGTACACG']


500 chuỗi ngẫu nhiên: khớp brute force

n = 10^5, mã 2 bit : 0.182s, 4448 chuỗi lặp
n = 10^5, set chuỗi: 0.146s, 4448 chuỗi lặp


# 9. Vì sao đúng? Độ phức tạp?

- Mã hóa 2 bit là **song ánh** giữa chuỗi 10 ký tự `ACGT` và số `0..2^20−1` → so số tương đương so chuỗi.
- Sau khi xử lý ký tự `i` (với `i >= 9`), `code` đúng bằng mã của `s[i−9..i]`: dịch trái 2 bit rồi `& MASK` loại đúng 2 bit của ký tự `s[i−10]`.
- `seen` ghi các cửa sổ đã gặp; `added` đảm bảo mỗi chuỗi lặp chỉ thêm một lần.

| Cách | Thời gian | Bộ nhớ |
|---|---|---|
| So mọi cặp | `O(n² · 10)` | `O(n)` |
| Set chuỗi | `O(10 · n)` | `O(10 · n)` |
| Mã 2 bit | `O(n)` | `O(n)` số nguyên (tối đa `2^20` giá trị) |

# 10. Những lỗi dễ mắc

## Lỗi 1: Trả về trùng lặp khi chuỗi xuất hiện ≥ 3 lần

In [19]:
def wrong_list(s):
    seen, out = set(), []
    for i in range(len(s) - 9):
        sub = s[i:i + 10]
        if sub in seen:
            out.append(sub)                        # SAI: thêm mỗi lần gặp lại
        seen.add(sub)
    return out

print("Sai :", wrong_list("AAAAAAAAAAAAA"), "| Đúng:", sol.findRepeatedDnaSequences("AAAAAAAAAAAAA"))

Sai : ['AAAAAAAAAA', 'AAAAAAAAAA', 'AAAAAAAAAA'] | Đúng: ['AAAAAAAAAA']


## Lỗi 2: Quên `& MASK`

In [20]:
def wrong_no_mask(s):
    seen, added, out, code = set(), set(), [], 0
    for i, ch in enumerate(s):
        code = (code << 2) | BITS[ch]             # SAI: code giữ cả các ký tự cũ, lớn dần mãi
        if i >= 9:
            if code in seen and code not in added:
                out.append(s[i - 9:i + 1]); added.add(code)
            seen.add(code)
    return out

print("Sai :", wrong_no_mask("AAAAACCCCCAAAAACCCCCCAAAAAGGGTTT"), "| Đúng:", sorted(sol.findRepeatedDnaSequences("AAAAACCCCCAAAAACCCCCCAAAAAGGGTTT")))

Sai :

 [] | Đúng: ['AAAAACCCCC', 'CCCCCAAAAA']


Không cắt bớt thì `code` mã hóa **toàn bộ tiền tố**, các cửa sổ không bao giờ trùng mã. (Riêng chuỗi toàn `A` mã luôn là 0 nên lỗi bị che!)

## Lỗi 3: Lệch chỉ số cửa sổ (`range(len(s) - 10)`)

In [21]:
def wrong_range(s):
    seen, rep = set(), set()
    for i in range(len(s) - 10):                   # SAI: bỏ mất cửa sổ cuối cùng
        sub = s[i:i + 10]
        (rep if sub in seen else seen).add(sub)
    return sorted(rep)

print("Sai :", wrong_range("AAAAAAAAAAA"), "| Đúng:", sol.findRepeatedDnaSequences("AAAAAAAAAAA"))

Sai : [] | Đúng: ['AAAAAAAAAA']


Chuỗi dài `n` có `n − 9` cửa sổ: `range(len(s) - 9)`.

## Lỗi 4: Dùng `hash(sub)` của Python làm khóa thay cho chuỗi

Hàm băm có thể **va chạm** (hai chuỗi khác nhau cùng giá trị băm) → báo sai chuỗi lặp. Mã 2 bit an toàn vì là song ánh; với bảng chữ cái lớn phải kiểm tra lại chuỗi khi trùng băm (Rabin–Karp, bài tập 5).

# 11. Mẹo nhận diện pattern

- **Mọi chuỗi con độ dài cố định `L`** → cửa sổ trượt; lưu vào set/dict để phát hiện lặp.
- **Bảng chữ cái nhỏ** → mã hóa mỗi ký tự bằng vài bit, cửa sổ thành một số (cập nhật bằng dịch bit + mặt nạ).
- **Bảng chữ cái lớn** → rolling hash: `h = (h · B − s[i−L] · B^L + s[i]) mod M`.

Bài liên quan: **#1044 Longest Duplicate Substring** (Hard), **#28 Find the Index of the First Occurrence** (Rabin–Karp), **#1461 Check If a String Contains All Binary Codes of Size K**, **#718**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Tính `encode` của `"CAT"` và cho biết sau khi trượt thêm ký tự `'G'` (cửa sổ 3 ký tự, `MASK = 2^6 − 1`) thì mã mới là gì.

## Bài 2 — Điền chỗ trống (bản hai set)

In [22]:
def exercise_2_repeated(s: str) -> List[str]:
    seen, repeated = set(), set()
    for i in range(len(s) - 9):
        # TODO
        pass
    return list(repeated)


# assert sorted(exercise_2_repeated("AAAAACCCCCAAAAACCCCCCAAAAAGGGTTT")) == ["AAAAACCCCC", "CCCCCAAAAA"]
# assert exercise_2_repeated("AAAAAAAAAAAAA") == ["AAAAAAAAAA"]
# assert exercise_2_repeated("ACG") == []

## Bài 3 — Biến thể: độ dài `L` bất kỳ, xuất hiện **ít nhất `k` lần**

Trả về danh sách đã sắp xếp.

In [23]:
def exercise_3_repeated_lk(s: str, L: int, k: int) -> List[str]:
    # TODO
    pass


# assert exercise_3_repeated_lk("ABABAB", 2, 3) == ["AB"]
# assert exercise_3_repeated_lk("ABABAB", 2, 2) == ["AB", "BA"]
# assert exercise_3_repeated_lk("AAAA", 5, 1) == []

## Bài 4 — Biến thể: Check If a String Contains All Binary Codes of Size K (LeetCode #1461)

Chuỗi nhị phân `s` có chứa **mọi** chuỗi nhị phân độ dài `k` làm chuỗi con không? Dùng mã `k` bit với cửa sổ trượt; đếm số mã khác nhau.

In [24]:
def exercise_4_has_all_codes(s: str, k: int) -> bool:
    # TODO
    pass


# assert exercise_4_has_all_codes("00110110", 2) is True
# assert exercise_4_has_all_codes("0110", 1) is True
# assert exercise_4_has_all_codes("0110", 2) is False

## Bài 5 — Mini challenge: rolling hash (Rabin–Karp) cho bảng chữ cái bất kỳ

Đếm số **chuỗi con khác nhau** độ dài `L` của `s` (chữ thường) bằng rolling hash `h = h·B + mã ký tự (mod M)`; khi trượt: trừ đi đóng góp `mã(s[i−L]) · B^L`. Để an toàn với va chạm, lưu `dict: hash → list các vị trí bắt đầu` và so chuỗi thật khi trùng hash.

In [25]:
def exercise_5_count_distinct(s: str, L: int) -> int:
    # TODO
    pass


# assert exercise_5_count_distinct("banana", 2) == 3
# assert exercise_5_count_distinct("aaaa", 2) == 1
# assert exercise_5_count_distinct("abc", 4) == 0

# 13. Gợi ý

**Bài 1**
- Mức 1: `C=01, A=00, T=11` → `010011` = 19; thêm `G=10`: `(19 << 2 | 2) & 63`.

**Bài 3**
- Mức 1: `Counter(s[i:i+L] for i in range(len(s) - L + 1))`, lọc `>= k`.

**Bài 4**
- Mức 1: cần đủ `2^k` mã khác nhau; `code = ((code << 1) | bit) & ((1 << k) − 1)`.

**Bài 5**
- Mức 1: `B = 131`, `M = 2^61 − 1`, `BL = pow(B, L, M)`.
- Mức 2: trượt: `h = (h * B + c_mới − c_cũ * BL) % M`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [26]:
c = encode("CAT")
print("encode('CAT') =", c, "=", format(c, "06b"))
c2 = ((c << 2) | BITS["G"]) & ((1 << 6) - 1)
print("sau khi thêm 'G':", c2, "=", format(c2, "06b"), "| encode('ATG') =", encode("ATG"))

encode('CAT') = 19 = 010011
sau khi thêm 'G': 14 = 001110 | encode('ATG') = 14


## Đáp án Bài 2

In [27]:
def answer_2_repeated(s: str) -> List[str]:
    return repeated_sets(s)

assert sorted(answer_2_repeated("AAAAACCCCCAAAAACCCCCCAAAAAGGGTTT")) == ["AAAAACCCCC", "CCCCCAAAAA"]
assert answer_2_repeated("AAAAAAAAAAAAA") == ["AAAAAAAAAA"]
assert answer_2_repeated("ACG") == []
print("Bài 2 PASS (giống repeated_sets ở mục 5)")

Bài 2 PASS (giống repeated_sets ở mục 5)


## Đáp án Bài 3

In [28]:
def answer_3_repeated_lk(s: str, L: int, k: int) -> List[str]:
    cnt = Counter(s[i:i + L] for i in range(len(s) - L + 1))
    return sorted(w for w, c in cnt.items() if c >= k)

assert answer_3_repeated_lk("ABABAB", 2, 3) == ["AB"]
assert answer_3_repeated_lk("ABABAB", 2, 2) == ["AB", "BA"]
assert answer_3_repeated_lk("AAAA", 5, 1) == []
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [29]:
def answer_4_has_all_codes(s: str, k: int) -> bool:
    need = 1 << k
    mask = need - 1
    seen, code = set(), 0
    for i, ch in enumerate(s):
        code = ((code << 1) | (ch == "1")) & mask
        if i >= k - 1:
            seen.add(code)
            if len(seen) == need:
                return True
    return False

assert answer_4_has_all_codes("00110110", 2) is True
assert answer_4_has_all_codes("0110", 1) is True
assert answer_4_has_all_codes("0110", 2) is False
for _ in range(2000):
    t = "".join(random.choice("01") for _ in range(random.randint(1, 14)))
    k = random.randint(1, 3)
    exp = len({t[i:i + k] for i in range(len(t) - k + 1)}) == 2 ** k
    assert answer_4_has_all_codes(t, k) == exp
print("Bài 4 PASS")

Bài 4 PASS

## Đáp án Bài 5

In [30]:
def answer_5_count_distinct(s: str, L: int) -> int:
    n = len(s)
    if L > n:
        return 0
    B, M = 131, (1 << 61) - 1
    BL = pow(B, L, M)
    h = 0
    buckets = {}                          # hash -> list vị trí bắt đầu của các chuỗi KHÁC NHAU
    distinct = 0
    for i, ch in enumerate(s):
        h = (h * B + ord(ch)) % M
        if i >= L:
            h = (h - ord(s[i - L]) * BL) % M
        if i >= L - 1:
            start = i - L + 1
            lst = buckets.setdefault(h, [])
            if not any(s[p:p + L] == s[start:start + L] for p in lst):   # so chuỗi thật khi trùng hash
                lst.append(start)
                distinct += 1
    return distinct

assert answer_5_count_distinct("banana", 2) == 3
assert answer_5_count_distinct("aaaa", 2) == 1
assert answer_5_count_distinct("abc", 4) == 0
for _ in range(2000):
    t = "".join(random.choice("abc") for _ in range(random.randint(1, 15)))
    L_ = random.randint(1, 5)
    assert answer_5_count_distinct(t, L_) == len({t[i:i + L_] for i in range(len(t) - L_ + 1)})
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Duyệt mọi cửa sổ dài 10; set `seen` để phát hiện lặp, set thứ hai để không trả trùng. Tối ưu: mã hóa `A, C, G, T` thành 2 bit, cửa sổ là số 20 bit cập nhật bằng `((code << 2) | bit) & 0xFFFFF` — `O(n)`, không va chạm.

---

## Checklist trước khi nộp

- [ ] Có `n − 9` cửa sổ (`range(len(s) - 9)`)?
- [ ] Mỗi chuỗi lặp chỉ xuất hiện một lần trong kết quả?
- [ ] Bản bit: có `& MASK`?
- [ ] `len(s) < 10` → `[]`?